# Physique Statistique & Thermodynamique appliquées aux réseaux VLS français

**Fossé R. & Pallares G. — 2025–2026**

---

Ce notebook explore comment des concepts fondamentaux de physique — entropie de Boltzmann, distribution statistique, diffusion, modèle de gravité, percolation — peuvent être appliqués à l'analyse des réseaux de vélos en libre-service (VLS) français. L'objectif n'est pas métaphorique : plusieurs de ces formalisme sont utilisés en *physique urbaine* (*urban physics*) et en science des réseaux de transport.

## Sommaire

1. Chargement et préparation des données
2. Entropie de Shannon — désordre de la distribution des capacités
3. Distribution de Boltzmann — modélisation statistique des capacités
4. Loi de puissance (loi d'échelle) — topologie du réseau
5. Modèle de gravité newtonien — flux inter-urbains
6. Diffusion — redistribution des vélos et équation de Fick
7. Percolation — seuil de connectivité du réseau
8. Principe d'entropie maximale — distribution optimale théorique
9. Synthèse thermodynamique : diagramme d'état des réseaux VLS


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from scipy import stats, optimize, spatial
from scipy.special import factorial
from scipy.stats import entropy as scipy_entropy
import networkx as nx
from pathlib import Path

# Style global
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FFFFFF',
    'axes.edgecolor':   '#CCCCCC',
    'axes.labelcolor':  '#333333',
    'text.color':       '#333333',
    'xtick.color':      '#555555',
    'ytick.color':      '#555555',
    'grid.color':       '#EEEEEE',
    'grid.linewidth':   0.8,
    'font.family':      'serif',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'legend.fontsize':  9,
})

BLUE   = '#1A6FBF'
ORANGE = '#D96B27'
RED    = '#C0392B'
GREEN  = '#1E8449'
GREY   = '#7F8C8D'

DATA_PATH = Path('../data/stations_gold_standard_final.parquet')
print('Dépendances chargées.')

---
## 1. Chargement et préparation des données

In [ ]:
df_raw = pd.read_parquet(DATA_PATH)
print(f'Dataset brut : {len(df_raw):,} lignes, {df_raw.shape[1]} colonnes')

# Filtres : stations dock-based avec capacité valide
_NON_CITY = {'France', 'FR', 'Grand Est', 'Basque Country'}

df = (
    df_raw
    .query('station_type == "docked_bike"')
    .dropna(subset=['capacity', 'lat', 'lon', 'city'])
    .query('capacity > 0')
    .loc[lambda x: ~x['city'].isin(_NON_CITY)]
    .copy()
)

print(f'Dataset filtré (dock-based, capacité>0) : {len(df):,} stations')
print(f'Villes distinctes : {df["city"].nunique()}')
print(f'Capacité totale nationale : {df["capacity"].sum():,} emplacements')

# Agrégation par ville
city_agg = (
    df.groupby('city')
    .agg(
        n_stations   = ('capacity', 'count'),
        total_cap    = ('capacity', 'sum'),
        mean_cap     = ('capacity', 'mean'),
        std_cap      = ('capacity', 'std'),
        lat_mean     = ('lat', 'mean'),
        lon_mean     = ('lon', 'mean'),
    )
    .query('n_stations >= 5')
    .reset_index()
)
print(f'\nVilles avec >= 5 stations : {len(city_agg)}')

---
## 2. Entropie de Shannon / Boltzmann

### Analogie physique

En thermodynamique statistique, l'entropie de Boltzmann est définie par :

$$S = -k_B \sum_i p_i \ln p_i$$

où $p_i$ est la probabilité d'occuper le micro-état $i$ et $k_B$ la constante de Boltzmann. Shannon (1948) a appliqué le même formalisme à l'information.

**Application bikeshare :** On modélise chaque station comme un micro-état. La *capacité* joue le rôle de l'*énergie*. La distribution des capacités au sein d'une ville définit une distribution de probabilité $\{p_i\}$ sur laquelle on calcule l'entropie. Une entropie élevée signifie un réseau *uniforme* (toutes les stations ont des capacités similaires) ; une entropie faible révèle une *hiérarchie* forte (quelques grandes stations dominent).

L'entropie maximale théorique pour $N$ stations est $S_{\max} = \ln N$ (distribution uniforme).
On définit donc l'**entropie normalisée** :

$$H = \frac{S}{S_{\max}} = \frac{-\sum_i p_i \ln p_i}{\ln N} \in [0, 1]$$

In [ ]:
def shannon_entropy(capacities: np.ndarray) -> tuple[float, float]:
    """Retourne (S_brute, H_normalisee) pour un tableau de capacités."""
    total = capacities.sum()
    if total == 0 or len(capacities) < 2:
        return 0.0, 0.0
    p = capacities / total                # distribution de probabilité
    p = p[p > 0]                          # éviter log(0)
    S = -np.sum(p * np.log(p))           # entropie de Shannon (naturelle)
    S_max = np.log(len(capacities))       # entropie maximale
    H = S / S_max if S_max > 0 else 0.0
    return float(S), float(H)

# Calcul par ville
entropy_rows = []
for city, grp in df.groupby('city'):
    if len(grp) < 5:
        continue
    caps = grp['capacity'].values.astype(float)
    S, H = shannon_entropy(caps)
    entropy_rows.append({
        'city': city,
        'n': len(grp),
        'S_brute': S,
        'H_norm': H,
        'mean_cap': caps.mean(),
        'cv': caps.std() / caps.mean() if caps.mean() > 0 else 0,  # coeff. de variation
    })

df_ent = pd.DataFrame(entropy_rows).sort_values('H_norm', ascending=False)

# --- Figure ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme global de H
axes[0].hist(df_ent['H_norm'], bins=25, color=BLUE, alpha=0.85, edgecolor='white')
axes[0].axvline(df_ent['H_norm'].median(), color=ORANGE, lw=2, ls='--',
                label=f'Médiane = {df_ent["H_norm"].median():.3f}')
axes[0].set_xlabel('Entropie normalisée H')
axes[0].set_ylabel('Nombre de villes')
axes[0].set_title('Distribution de l\'entropie de capacité\n(réseaux VLS français, dock-based)')
axes[0].legend()
axes[0].grid(True)

# H vs coefficient de variation
sc = axes[1].scatter(
    df_ent['cv'], df_ent['H_norm'],
    c=df_ent['n'], cmap='Blues', s=60, alpha=0.8,
    norm=mcolors.LogNorm(vmin=5, vmax=df_ent['n'].max()),
    edgecolors='grey', linewidths=0.4
)
plt.colorbar(sc, ax=axes[1], label='Nombre de stations')

# Annoter les extrêmes
for _, row in df_ent.nlargest(3, 'H_norm').iterrows():
    axes[1].annotate(row['city'], (row['cv'], row['H_norm']),
                     xytext=(5, 3), textcoords='offset points', fontsize=8)
for _, row in df_ent.nsmallest(3, 'H_norm').iterrows():
    axes[1].annotate(row['city'], (row['cv'], row['H_norm']),
                     xytext=(5, -10), textcoords='offset points', fontsize=8)

axes[1].set_xlabel('Coefficient de variation des capacités (σ/μ)')
axes[1].set_ylabel('Entropie normalisée H')
axes[1].set_title('Entropie vs hétérogénéité des capacités')
axes[1].grid(True)

fig.suptitle('Figure 2.1 — Entropie de Shannon des réseaux VLS', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Entropie H médiane : {df_ent["H_norm"].median():.3f}')
print(f'H max (réseau le plus uniforme) : {df_ent["H_norm"].max():.3f} — {df_ent.iloc[0]["city"]}')
print(f'H min (réseau le plus hiérarchisé) : {df_ent["H_norm"].min():.3f} — {df_ent.iloc[-1]["city"]}')

### 2.1 Entropie de Boltzmann et « température » d'un réseau

En physique statistique, la **température** $T$ est liée à la dérivée de l'entropie par rapport à l'énergie : $\frac{1}{T} = \frac{\partial S}{\partial E}$. On peut définir par analogie une **température de réseau** $T^* = \langle c \rangle / c_{\max}$, où $\langle c \rangle$ est la capacité moyenne et $c_{\max}$ la capacité maximale. Un réseau *chaud* ($T^* \to 1$) est un réseau où toutes les stations sont de grande capacité ; un réseau *froid* ($T^* \to 0$) a une grande station qui domine.

In [ ]:
temp_rows = []
for city, grp in df.groupby('city'):
    if len(grp) < 5:
        continue
    caps = grp['capacity'].values.astype(float)
    S, H = shannon_entropy(caps)
    T_star = caps.mean() / caps.max()   # température réduite
    temp_rows.append({'city': city, 'n': len(grp), 'T_star': T_star, 'H_norm': H,
                      'total_cap': caps.sum()})

df_temp = pd.DataFrame(temp_rows)

fig, ax = plt.subplots(figsize=(9, 6))

sc = ax.scatter(
    df_temp['T_star'], df_temp['H_norm'],
    c=df_temp['total_cap'], cmap='YlOrRd', s=70, alpha=0.85,
    norm=mcolors.LogNorm(vmin=50, vmax=df_temp['total_cap'].max()),
    edgecolors='grey', linewidths=0.4
)
plt.colorbar(sc, ax=ax, label='Capacité totale (emplacements)')

# Quadrants
ax.axhline(0.85, color=GREY, ls=':', lw=1)
ax.axvline(0.55, color=GREY, ls=':', lw=1)
ax.text(0.25, 0.90, 'Réseau uniforme\n& hiérarchisé', ha='center', fontsize=8, color=GREY)
ax.text(0.75, 0.90, 'Réseau uniforme\n& homogène', ha='center', fontsize=8, color=GREEN)
ax.text(0.25, 0.65, 'Réseau hiérarchisé\n& froid', ha='center', fontsize=8, color=RED)
ax.text(0.75, 0.65, 'Réseau dispersé\n& chaud', ha='center', fontsize=8, color=ORANGE)

for _, row in df_temp.nlargest(5, 'total_cap').iterrows():
    ax.annotate(row['city'], (row['T_star'], row['H_norm']),
                xytext=(6, 3), textcoords='offset points', fontsize=8, color=RED)

ax.set_xlabel('Température réduite T* = ⟨c⟩ / c_max')
ax.set_ylabel('Entropie normalisée H')
ax.set_title('Diagramme Entropie–Température des réseaux VLS\n(analogie thermodynamique)')
ax.grid(True)
plt.tight_layout()
plt.savefig('../data/processed/fig_entropy_temperature.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Distribution de Boltzmann

### Analogie physique

La distribution de Maxwell-Boltzmann décrit la probabilité qu'une particule d'un gaz parfait ait une énergie $E$ à l'équilibre thermique :

$$P(E) \propto g(E) \, e^{-E / k_B T}$$

où $g(E)$ est la *dégénérescence* (nombre d'états d'énergie $E$).

**Application bikeshare :** La capacité d'une station joue le rôle de l'énergie. Si les stations étaient allouées à l'équilibre thermique (principe de moindre action, maximisation de l'entropie sous contrainte d'énergie totale fixe), leur distribution suivrait une loi exponentielle décroissante. Nous testons cette hypothèse sur les données nationales et par réseau.

In [ ]:
caps_all = df['capacity'].values.astype(float)

# Ajustement exponentiel (Boltzmann) : P(c) = lambda * exp(-lambda * c)
loc_exp, scale_exp = stats.expon.fit(caps_all, floc=0)
lambda_fit = 1.0 / scale_exp

# Ajustement lognormal
s_ln, loc_ln, scale_ln = stats.lognorm.fit(caps_all, floc=0)

# Ajustement Gamma
a_gam, loc_gam, scale_gam = stats.gamma.fit(caps_all, floc=0)

# --- Figure ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme + fits
bins = np.linspace(0, 80, 50)
counts, edges, _ = axes[0].hist(caps_all, bins=bins, density=True,
                                 color=BLUE, alpha=0.6, label='Données observées')
x_fit = np.linspace(1, 80, 300)
axes[0].plot(x_fit, stats.expon.pdf(x_fit, loc_exp, scale_exp),
             color=ORANGE, lw=2, label=f'Boltzmann exponentiel (λ={lambda_fit:.3f})')
axes[0].plot(x_fit, stats.lognorm.pdf(x_fit, s_ln, loc_ln, scale_ln),
             color=GREEN, lw=2, ls='--', label='Lognormal')
axes[0].plot(x_fit, stats.gamma.pdf(x_fit, a_gam, loc_gam, scale_gam),
             color=RED, lw=2, ls=':', label=f'Gamma (α={a_gam:.2f})')

axes[0].set_xlabel('Capacité (emplacements par station)')
axes[0].set_ylabel('Densité de probabilité')
axes[0].set_title('Ajustement de distribution de Boltzmann\n(capacités nationales)')
axes[0].legend()
axes[0].set_xlim(0, 80)
axes[0].grid(True)

# Q-Q plot exponentiel
samp_sorted = np.sort(caps_all[caps_all <= 80])
n = len(samp_sorted)
q_theor = stats.expon.ppf(np.linspace(0.01, 0.99, n), loc_exp, scale_exp)
q_obs   = np.quantile(samp_sorted, np.linspace(0.01, 0.99, n))
axes[1].scatter(q_theor, q_obs, color=BLUE, s=5, alpha=0.4)
lim = max(q_theor.max(), q_obs.max())
axes[1].plot([0, lim], [0, lim], color=ORANGE, lw=2, ls='--', label='Idéal Boltzmann')
axes[1].set_xlabel('Quantiles théoriques (exponentiel)')
axes[1].set_ylabel('Quantiles observés')
axes[1].set_title('Q-Q plot : données vs distribution de Boltzmann')
axes[1].legend()
axes[1].grid(True)

fig.suptitle('Figure 3.1 — Test de la distribution de Boltzmann sur les capacités VLS', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Tests statistiques
ks_exp = stats.kstest(caps_all[caps_all <= 100], 'expon', args=(loc_exp, scale_exp))
ks_ln  = stats.kstest(caps_all[caps_all <= 100], 'lognorm', args=(s_ln, loc_ln, scale_ln))
print(f'KS test Boltzmann exponentiel : statistic={ks_exp.statistic:.4f}, p={ks_exp.pvalue:.4f}')
print(f'KS test Lognormal             : statistic={ks_ln.statistic:.4f}, p={ks_ln.pvalue:.4f}')
print(f'\nConclusion : la distribution lognormale (p={ks_ln.pvalue:.4f}) est '
      f'{"meilleure" if ks_ln.pvalue > ks_exp.pvalue else "moins bonne"} '
      f'que le modèle de Boltzmann exponentiel.')
print(f'Température thermodynamique analogue : T* = 1/λ = {1/lambda_fit:.2f} emplacements')

---
## 4. Loi de puissance — Topologie du réseau (physique des systèmes complexes)

### Analogie physique

Les réseaux *scale-free* (invariants d'échelle) suivent une distribution de degré en loi de puissance :

$$P(k) \sim k^{-\gamma}$$

Ce comportement est analogue aux systèmes critiques en physique (transitions de phase, phénomènes de percolation). Barabási & Albert (1999) ont montré que de nombreux réseaux réels (Internet, réseaux de citations, réseaux sociaux) sont scale-free. Les réseaux de transport peuvent exhiber ce comportement au niveau des capacités.

**Application :** On teste si la distribution des capacités suit une loi de puissance en utilisant la méthode du maximum de vraisemblance (Clauset et al., 2009) et en comparant les ajustements log-log.

In [ ]:
from scipy.optimize import curve_fit

# CCDF (Complementary CDF) — méthode robuste pour tester les lois de puissance
caps_sorted = np.sort(caps_all)[::-1]
n_total = len(caps_sorted)
ccdf = np.arange(1, n_total + 1) / n_total

# Seuil minimal pour la loi de puissance
c_min = 10  # emplacements minimum
mask = caps_sorted >= c_min
caps_pw = caps_sorted[mask]
ccdf_pw = ccdf[mask]

# Ajustement loi de puissance : CCDF(c) = A * c^(-alpha+1)
def power_law_ccdf(c, alpha, A):
    return A * c ** (-(alpha - 1))

popt, _ = curve_fit(power_law_ccdf, caps_pw, ccdf_pw, p0=[2.0, 100.0], maxfev=5000)
alpha_fit, A_fit = popt

# Estimation MLE de l'exposant (Clauset 2009)
alpha_mle = 1 + n_total / np.sum(np.log(caps_all[caps_all >= c_min] / (c_min - 0.5)))

# --- Figure ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CCDF en log-log
ax = axes[0]
ax.loglog(caps_sorted, ccdf, '.', color=BLUE, ms=2, alpha=0.4, label='Données observées')
c_range = np.linspace(c_min, caps_sorted.max(), 200)
ax.loglog(c_range, power_law_ccdf(c_range, alpha_fit, A_fit),
          color=ORANGE, lw=2, label=f'Loi de puissance α={alpha_fit:.2f}')
ax.loglog(c_range, stats.expon.sf(c_range, loc_exp, scale_exp),
          color=GREEN, lw=2, ls='--', label='Boltzmann exponentiel')
ax.set_xlabel('Capacité c (emplacements)')
ax.set_ylabel('P(C ≥ c) — CCDF')
ax.set_title('Loi de puissance vs Boltzmann\n(CCDF, échelle log-log)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

# Distribution des tailles de réseau par ville
ax2 = axes[1]
n_stations_cities = city_agg['n_stations'].values
ns_sorted = np.sort(n_stations_cities)[::-1]
ccdf_city = np.arange(1, len(ns_sorted) + 1) / len(ns_sorted)
ax2.loglog(ns_sorted, ccdf_city, 'o', color=BLUE, ms=6, alpha=0.7, label='Villes observées')

# Fit
mask2 = ns_sorted >= 10
if mask2.sum() > 5:
    popt2, _ = curve_fit(power_law_ccdf, ns_sorted[mask2], ccdf_city[mask2],
                         p0=[2.0, 1.0], maxfev=5000)
    c2 = np.linspace(10, ns_sorted.max(), 100)
    ax2.loglog(c2, power_law_ccdf(c2, popt2[0], popt2[1]),
               color=RED, lw=2, label=f'Loi de puissance α={popt2[0]:.2f}')

# Annoter les grandes villes
top5 = city_agg.nlargest(5, 'n_stations')
for _, row in top5.iterrows():
    rank = np.searchsorted(-ns_sorted, -row['n_stations']) + 1
    ax2.annotate(row['city'], (row['n_stations'], ccdf_city[rank-1]),
                 xytext=(5, 3), textcoords='offset points', fontsize=8)

ax2.set_xlabel('Nombre de stations par ville')
ax2.set_ylabel('P(N ≥ n) — CCDF')
ax2.set_title('Distribution de la taille des réseaux\n(log-log, analogie Zipf / loi de puissance)')
ax2.legend()
ax2.grid(True, which='both', alpha=0.3)

fig.suptitle('Figure 4.1 — Loi de puissance et invariance d\'échelle des réseaux VLS', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Exposant loi de puissance (CCDF fit) : α = {alpha_fit:.3f}')
print(f'Exposant MLE (Clauset 2009)          : α = {alpha_mle:.3f}')
print(f'Réseaux scale-free typiques : α ∈ [2, 3] → α={alpha_mle:.2f} ', end='')
print('(dans la zone scale-free)' if 2 <= alpha_mle <= 3 else '(hors zone scale-free typique)')

---
## 5. Modèle de gravité newtonien — Flux inter-urbains

### Analogie physique

La loi universelle de la gravitation de Newton stipule :

$$F_{ij} = G \frac{m_i \cdot m_j}{d_{ij}^2}$$

Le **modèle de gravité** de Reilly (1929) / Zipf (1946) applique ce principe aux flux de mobilité :

$$T_{ij} = K \frac{P_i^{\alpha} \cdot P_j^{\beta}}{d_{ij}^{\gamma}}$$

où $T_{ij}$ est le flux entre les villes $i$ et $j$, $P$ la masse (ici : capacité totale du réseau VLS), et $d_{ij}$ la distance.

**Application :** On utilise la capacité totale comme masse et on calcule la force gravitationnelle entre chaque paire de villes. Les paires à forte force gravitationnelle sont les candidats naturels à des systèmes de vélo intercommunaux.

In [ ]:
from itertools import combinations

cities_top = city_agg.nlargest(30, 'total_cap').copy()

def haversine_km(lat1, lon1, lat2, lon2):
    """Distance haversine en km."""
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Matrice de force gravitationnelle (G = 1 par convention)
grav_rows = []
for i, j in combinations(cities_top.index, 2):
    ri, rj = cities_top.loc[i], cities_top.loc[j]
    d = haversine_km(ri['lat_mean'], ri['lon_mean'], rj['lat_mean'], rj['lon_mean'])
    if d < 1:
        continue
    F = (ri['total_cap'] * rj['total_cap']) / d**2
    grav_rows.append({'city_i': ri['city'], 'city_j': rj['city'],
                      'dist_km': d, 'F_grav': F,
                      'mass_i': ri['total_cap'], 'mass_j': rj['total_cap']})

df_grav = pd.DataFrame(grav_rows).sort_values('F_grav', ascending=False)

# --- Figure ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Top 15 paires gravitationnelles
top15 = df_grav.head(15).copy()
top15['paire'] = top15['city_i'] + ' — ' + top15['city_j']
top15 = top15.sort_values('F_grav')

bars = axes[0].barh(top15['paire'], top15['F_grav'], color=BLUE, alpha=0.8)
axes[0].set_xlabel('Force gravitationnelle F = M₁·M₂/d² (u.a.)')
axes[0].set_title('Top 15 paires de villes\npar force gravitationnelle VLS')
axes[0].grid(True, axis='x')

# F vs distance (loi d'échelle)
axes[1].loglog(df_grav['dist_km'], df_grav['F_grav'], '.', color=BLUE, ms=4, alpha=0.4)

# Régression log-log
log_d = np.log(df_grav['dist_km'])
log_F = np.log(df_grav['F_grav'])
slope, intercept, r, p, _ = stats.linregress(log_d, log_F)
d_range = np.exp(np.linspace(log_d.min(), log_d.max(), 100))
axes[1].loglog(d_range, np.exp(intercept) * d_range**slope, color=ORANGE, lw=2,
               label=f'F ∝ d^{slope:.2f}  (R²={r**2:.3f})')
axes[1].set_xlabel('Distance inter-urbaine (km)')
axes[1].set_ylabel('Force gravitationnelle (u.a.)')
axes[1].set_title('Force gravitationnelle vs distance\n(loi inverse de la distance)')
axes[1].legend()
axes[1].grid(True, which='both', alpha=0.3)
axes[1].text(0.05, 0.05, f'Exposant attendu : -2.0\nObservé : {slope:.2f}',
             transform=axes[1].transAxes, bbox=dict(fc='white', alpha=0.7), fontsize=9)

fig.suptitle('Figure 5.1 — Modèle de gravité newtonien appliqué aux réseaux VLS', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Top 5 paires gravitationnelles :')
print(df_grav[['city_i', 'city_j', 'dist_km', 'F_grav']].head().to_string(index=False))
print(f'\nExposant distance observé : {slope:.3f} (Newton : -2.00)')

---
## 6. Diffusion — Redistribution des vélos et équation de Fick

### Analogie physique

La première loi de Fick décrit la diffusion d'un soluté le long d'un gradient de concentration :

$$\vec{J} = -D \, \nabla c$$

où $\vec{J}$ est le flux de matière, $D$ le coefficient de diffusion, et $c$ la concentration locale.

**Application :** La *densité de capacité* d'une station (emplacements/km²) est analogue à une concentration. Le gradient de densité entre deux stations voisines prédit la direction et l'intensité du flux de redistribution des vélos. Un opérateur rationnel devrait, comme les lois de la diffusion, équilibrer les concentrations (rééquilibrer les stations surchargées vers les sous-chargées).

On calcule le *gradient de Fick spatial* entre stations voisines et on compare à l'*asymétrie réelle* des flux (données Montpellier disponibles).

In [ ]:
# Cas d'étude : Montpellier (données de flux disponibles)
df_mtpl = df[df['city'].str.contains('Montpellier', na=False)].copy()
print(f'Stations Montpellier : {len(df_mtpl)}')

# Construire une grille spatiale et calculer la densité locale de capacité
from scipy.spatial import cKDTree

coords_rad = np.radians(df_mtpl[['lat', 'lon']].values)
caps_mtpl = df_mtpl['capacity'].values.astype(float)

# Pour chaque station, calculer la densité locale = capacité / surface (dans rayon R)
R_km = 0.5  # rayon de voisinage (km)
R_deg = R_km / 111.0   # approximation

coords_deg = df_mtpl[['lat', 'lon']].values
tree = cKDTree(coords_deg)

densities = []
gradients = []
for i, (pt, c_i) in enumerate(zip(coords_deg, caps_mtpl)):
    idxs = tree.query_ball_point(pt, R_deg)
    idxs = [j for j in idxs if j != i]
    if not idxs:
        densities.append(c_i)
        gradients.append(0.0)
        continue
    # Densité locale : somme des capacités voisines / aire du disque
    local_density = caps_mtpl[idxs].sum() / (np.pi * R_km**2)
    densities.append(local_density)
    # Gradient (différence avec la moyenne des voisins)
    mean_neighbor_cap = caps_mtpl[idxs].mean()
    gradients.append(c_i - mean_neighbor_cap)  # positif = station plus grande que ses voisines

df_mtpl = df_mtpl.copy()
df_mtpl['local_density'] = densities
df_mtpl['fick_gradient'] = gradients  # F_Fick = -D * gradient
df_mtpl['fick_flux'] = -df_mtpl['fick_gradient']  # sens du flux Fick prédit

# --- Figure ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Carte des densités locales
sc1 = axes[0].scatter(df_mtpl['lon'], df_mtpl['lat'],
                       c=df_mtpl['local_density'], cmap='Blues', s=50, alpha=0.85)
plt.colorbar(sc1, ax=axes[0], label='Densité locale (empl/km²)')
axes[0].set_title('Densité locale de capacité\n(concentration — analogie Fick)')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].grid(True, alpha=0.3)

# Gradient de Fick
cmap_div = plt.cm.RdBu_r
vmax = np.abs(df_mtpl['fick_gradient']).quantile(0.95)
sc2 = axes[1].scatter(df_mtpl['lon'], df_mtpl['lat'],
                       c=df_mtpl['fick_gradient'],
                       cmap=cmap_div, vmin=-vmax, vmax=vmax, s=50, alpha=0.85)
plt.colorbar(sc2, ax=axes[1], label='Gradient Fick (empl)')
axes[1].set_title('Gradient de diffusion de Fick\n(bleu = source, rouge = puits)')
axes[1].set_xlabel('Longitude')
axes[1].grid(True, alpha=0.3)

# Distribution des gradients
axes[2].hist(df_mtpl['fick_gradient'], bins=30, color=BLUE, alpha=0.8, edgecolor='white')
axes[2].axvline(0, color=ORANGE, lw=2, ls='--', label='Équilibre de Fick')
axes[2].axvline(df_mtpl['fick_gradient'].mean(), color=RED, lw=2,
                label=f'Moyenne = {df_mtpl["fick_gradient"].mean():.1f}')
axes[2].set_xlabel('Gradient de Fick (empl — empl voisins moyen)')
axes[2].set_ylabel('Nombre de stations')
axes[2].set_title('Distribution des gradients de diffusion\n(symétrie = équilibre thermique)')
axes[2].legend()
axes[2].grid(True)

fig.suptitle('Figure 6.1 — Diffusion de Fick appliquée aux capacités VLS (Montpellier)', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Coefficient de diffusion estimé
D_analogue = caps_mtpl.var() / (2 * R_km**2)  # analogue à k_B T via la relation d'Einstein
print(f'Gradient de Fick moyen : {np.mean(gradients):.2f} emplacements')
print(f'Déséquilibre diffusif (|gradient| médian) : {np.median(np.abs(gradients)):.2f} emplacements')
print(f'Coefficient de diffusion analogue D* : {D_analogue:.2f} (empl²/km²)')
print(f'\nRelation d\'Einstein : D* = σ²/(2t) → variance spatiale des capacités = {caps_mtpl.var():.2f}')

---
## 7. Percolation — Seuil de connectivité du réseau

### Analogie physique

La **théorie de la percolation** (Broadbent & Hammersley, 1957) étudie la formation de chemins continus dans un réseau aléatoire. Il existe un **seuil critique** $p_c$ au-dessus duquel un *cluster géant* percolant apparaît soudainement — analogue à une transition de phase.

En physique des matériaux, ce seuil correspond par exemple à la transition isolant-conducteur. Dans les réseaux de transport, il définit le seuil de *couverture* : au-delà d'une certaine densité de stations, le réseau devient effectivement continu (un usager peut toujours trouver une station à distance de marche).

**Application :** On construit un graphe de voisinage (deux stations sont connectées si $d \leq r$) et on étudie la taille du *composante géante* en fonction du rayon $r$. Le seuil de percolation $r_c$ révèle la *portée critique d'accessibilité à pied*.

In [ ]:
# Percolation sur Montpellier
coords_mtpl = df_mtpl[['lat', 'lon']].values
n_stations_mtpl = len(coords_mtpl)

radii_km = np.linspace(0.05, 2.0, 60)  # de 50m à 2km

giant_fracs = []
mean_cluster = []
n_clusters   = []

for r_km in radii_km:
    r_deg = r_km / 111.0
    # Construire le graphe
    G = nx.Graph()
    G.add_nodes_from(range(n_stations_mtpl))
    tree_mtpl = cKDTree(coords_mtpl)
    pairs = tree_mtpl.query_pairs(r_deg)
    G.add_edges_from(pairs)
    
    comps = sorted(nx.connected_components(G), key=len, reverse=True)
    giant = len(comps[0]) / n_stations_mtpl if comps else 0
    giant_fracs.append(giant)
    sizes = [len(c) for c in comps]
    mean_cluster.append(np.mean(sizes))
    n_clusters.append(len(comps))

giant_fracs = np.array(giant_fracs)
mean_cluster = np.array(mean_cluster)

# Seuil de percolation : dérivée maximale de la fraction géante
deriv = np.gradient(giant_fracs, radii_km)
r_c_idx = np.argmax(deriv)
r_c = radii_km[r_c_idx]

# --- Figure ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Fraction géante vs rayon
axes[0].plot(radii_km * 1000, giant_fracs, color=BLUE, lw=2)
axes[0].axvline(r_c * 1000, color=ORANGE, lw=2, ls='--',
                label=f'Seuil de percolation r_c = {r_c*1000:.0f} m')
axes[0].axhline(0.5, color=GREY, lw=1, ls=':')
axes[0].set_xlabel('Rayon de voisinage (m)')
axes[0].set_ylabel('Fraction de la composante géante')
axes[0].set_title('Transition de percolation\n(Montpellier — Vélomagg)')
axes[0].legend()
axes[0].grid(True)

# Dérivée (susceptibilité — analogue à la chaleur spécifique)
axes[1].plot(radii_km * 1000, deriv, color=RED, lw=2)
axes[1].axvline(r_c * 1000, color=ORANGE, lw=2, ls='--', label=f'Maximum à r={r_c*1000:.0f} m')
axes[1].set_xlabel('Rayon de voisinage (m)')
axes[1].set_ylabel('dP_giant / dr (susceptibilité)')
axes[1].set_title('Susceptibilité du réseau\n(analogue à la chaleur spécifique en physique)')
axes[1].legend()
axes[1].grid(True)

# Nombre de clusters
axes[2].plot(radii_km * 1000, n_clusters, color=GREEN, lw=2)
axes[2].axvline(r_c * 1000, color=ORANGE, lw=2, ls='--', label=f'r_c = {r_c*1000:.0f} m')
axes[2].set_xlabel('Rayon de voisinage (m)')
axes[2].set_ylabel('Nombre de composantes connexes')
axes[2].set_title('Fragmentation du réseau vs rayon')
axes[2].legend()
axes[2].grid(True)

fig.suptitle('Figure 7.1 — Transition de phase par percolation — Seuil d\'accessibilité VLS (Montpellier)',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Seuil de percolation r_c = {r_c*1000:.0f} m')
print(f'A r_c, fraction géante = {giant_fracs[r_c_idx]:.3f}')
print(f'Interprétation : un usager marchant à pied {r_c*1000:.0f} m peut atteindre '
      f'{giant_fracs[r_c_idx]*100:.1f}% du réseau Vélomagg.')

---
## 8. Principe d'entropie maximale (MaxEnt)

### Analogie physique

Le **principe d'entropie maximale** (Jaynes, 1957) est un principe variationnel fondamental de la physique statistique : parmi toutes les distributions compatibles avec les contraintes observées, la nature choisit celle qui maximise l'entropie. C'est l'origine de la distribution de Boltzmann.

**Contrainte** : capacité totale fixe $C_{\text{tot}} = \sum_i c_i$  
**Résultat** : la distribution qui maximise $S = -\sum p_i \ln p_i$ sous contrainte $\sum p_i c_i = \langle c \rangle$ est une **distribution exponentielle** :

$$p^*(c) = \lambda e^{-\lambda c}, \quad \lambda = 1/\langle c \rangle$$

**Question :** La distribution réelle des capacités s'écarte-t-elle de l'optimum MaxEnt ? L'écart mesure l'*information ajoutée* par des contraintes urbanistiques (taille des plots, emplacements historiques, contraintes budgétaires).

In [ ]:
def maxent_excess(caps: np.ndarray) -> dict:
    """Calcule l'écart au principe d'entropie maximale."""
    caps = caps[caps > 0]
    n = len(caps)
    if n < 3:
        return {}
    
    # Distribution observée
    mean_c = caps.mean()
    p_obs = caps / caps.sum()
    S_obs = -np.sum(p_obs * np.log(p_obs))
    
    # Distribution MaxEnt (exponentielle) sous contrainte de capacité moyenne
    lam = 1.0 / mean_c
    # Discrétiser sur les mêmes valeurs de capacité
    c_vals = np.sort(np.unique(caps))
    p_maxent_raw = lam * np.exp(-lam * c_vals)
    p_maxent = p_maxent_raw / p_maxent_raw.sum()  # normalisation
    
    # Entropie MaxEnt théorique (continue) : S* = 1 + ln(1/lambda)
    S_maxent_theory = 1.0 + np.log(1.0 / lam)
    
    # KL divergence : D_KL(P_obs || P_maxent)
    # Histogramme des capacités observées
    bins = np.arange(caps.min(), caps.max() + 2)
    hist_obs, _ = np.histogram(caps, bins=bins, density=True)
    hist_maxent = lam * np.exp(-lam * bins[:-1])
    hist_maxent = hist_maxent / hist_maxent.sum() * (bins[1] - bins[0])  # normaliser comme densité
    
    # Filtrer les zéros
    mask = (hist_obs > 0) & (hist_maxent > 0)
    if mask.sum() < 2:
        return {}
    kl_div = np.sum(hist_obs[mask] * np.log(hist_obs[mask] / hist_maxent[mask]))
    
    return {
        'S_obs': S_obs,
        'S_maxent': S_maxent_theory,
        'S_ratio': S_obs / S_maxent_theory if S_maxent_theory > 0 else 0,
        'KL_divergence': kl_div,
        'mean_cap': mean_c,
        'n': n,
    }

maxent_rows = []
for city, grp in df.groupby('city'):
    if len(grp) < 10:
        continue
    res = maxent_excess(grp['capacity'].values)
    if res:
        res['city'] = city
        maxent_rows.append(res)

df_maxent = pd.DataFrame(maxent_rows).sort_values('KL_divergence', ascending=False)

# --- Figure ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution nationale : observé vs MaxEnt
ax = axes[0]
mean_c_nat = caps_all.mean()
lam_nat = 1.0 / mean_c_nat
bins_nat = np.arange(1, 80)
ax.hist(caps_all, bins=np.arange(0.5, 80.5), density=True,
        color=BLUE, alpha=0.6, label='Distribution observée')
c_range_nat = np.linspace(1, 79, 300)
ax.plot(c_range_nat, lam_nat * np.exp(-lam_nat * c_range_nat), color=ORANGE, lw=2,
        label=f'MaxEnt (λ=1/⟨c⟩={lam_nat:.3f})')
ax.set_xlabel('Capacité (emplacements)')
ax.set_ylabel('Densité')
ax.set_title('MaxEnt national : observé vs optimal')
ax.legend(); ax.set_xlim(0, 80); ax.grid(True)

# KL Divergence par ville (top 20)
top20 = df_maxent.head(20).sort_values('KL_divergence')
colors_bar = [ORANGE if kl > df_maxent['KL_divergence'].median() else BLUE
              for kl in top20['KL_divergence']]
axes[1].barh(top20['city'], top20['KL_divergence'], color=colors_bar, alpha=0.85)
axes[1].axvline(df_maxent['KL_divergence'].median(), color=GREY, ls='--', lw=1,
                label='Médiane')
axes[1].set_xlabel('Divergence KL (nats) — écart au MaxEnt')
axes[1].set_title('Top 20 villes par écart\nau principe d\'entropie maximale')
axes[1].legend(); axes[1].grid(True, axis='x')

# S_obs / S_maxent vs taille du réseau
sc = axes[2].scatter(df_maxent['n'], df_maxent['S_ratio'],
                      c=df_maxent['KL_divergence'], cmap='YlOrRd', s=60, alpha=0.8)
plt.colorbar(sc, ax=axes[2], label='Divergence KL')
axes[2].axhline(1.0, color=ORANGE, lw=2, ls='--', label='MaxEnt parfait')
axes[2].set_xscale('log')
axes[2].set_xlabel('Nombre de stations (log)')
axes[2].set_ylabel('S_obs / S_MaxEnt')
axes[2].set_title('Efficacité entropique vs taille du réseau')
axes[2].legend(); axes[2].grid(True)

fig.suptitle('Figure 8.1 — Principe d\'entropie maximale (Jaynes 1957) appliqué aux réseaux VLS',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'KL divergence médiane nationale : {df_maxent["KL_divergence"].median():.4f} nats')
print(f'Ville la plus proche du MaxEnt  : {df_maxent.nsmallest(1, "KL_divergence").iloc[0]["city"]}')
print(f'Ville la plus éloignée du MaxEnt: {df_maxent.nlargest(1, "KL_divergence").iloc[0]["city"]}')

---
## 9. Synthèse thermodynamique — Diagramme d'état des réseaux VLS

### Analogie avec un diagramme de phases

En thermodynamique, un **diagramme de phases** représente les états d'équilibre d'un système en fonction de ses variables d'état (température $T$, pression $P$, volume $V$). Les transitions de phase (liquide/gaz, ordre/désordre) correspondent à des discontinuités.

On construit un **diagramme d'état des réseaux VLS** avec :
- **Axe X ("Pression")** : densité de stations (stations/km² estimée) — pression de la demande sur le territoire
- **Axe Y ("Température")** : coefficient de variation des capacités (désordre thermique)
- **Couleur ("Énergie libre")** : entropie normalisée $H$
- **Taille** : capacité totale (énergie interne)

On cherche des analogues aux *phases* du système : réseau *gazeux* (stations dispersées, hétérogènes), réseau *liquide* (densité modérée, distribution modérée), réseau *solide* (dense, structuré, uniforme).

In [ ]:
# Construire le diagramme d'état
# Estimation de la superficie par convex hull des stations
from scipy.spatial import ConvexHull
from math import radians, cos

def city_area_km2(grp: pd.DataFrame) -> float:
    """Estime l'aire du réseau via convex hull des stations (km²)."""
    if len(grp) < 4:
        return np.nan
    lats = grp['lat'].values
    lons = grp['lon'].values
    # Convertir en coordonnées métriques (approximation locale)
    lat_mean = lats.mean()
    x = lons * 111.0 * cos(radians(lat_mean))
    y = lats * 111.0
    pts = np.column_stack([x, y])
    try:
        hull = ConvexHull(pts)
        return hull.volume  # en km² (2D → volume = aire)
    except Exception:
        return np.nan

state_rows = []
for city, grp in df.groupby('city'):
    if len(grp) < 5:
        continue
    caps = grp['capacity'].values.astype(float)
    area = city_area_km2(grp)
    if np.isnan(area) or area < 0.01:
        continue
    
    S, H = shannon_entropy(caps)
    cv = caps.std() / caps.mean() if caps.mean() > 0 else 0
    density = len(grp) / area  # stations/km² — analogie pression
    total_cap = caps.sum()
    
    state_rows.append({
        'city': city,
        'n': len(grp),
        'area_km2': area,
        'density': density,       # Pression P
        'cv': cv,                 # Température T
        'H_norm': H,              # Entropie
        'total_cap': total_cap,   # Énergie interne
        'mean_cap': caps.mean(),
    })

df_state = pd.DataFrame(state_rows)

# --- Diagramme de phases ---
fig, ax = plt.subplots(figsize=(12, 8))

sc = ax.scatter(
    np.log1p(df_state['density']),   # log(P)
    df_state['cv'],                   # T
    c=df_state['H_norm'],
    cmap='RdYlBu',
    s=df_state['total_cap'] / df_state['total_cap'].max() * 500 + 30,
    alpha=0.82,
    edgecolors='grey', linewidths=0.4
)

cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
cbar.set_label('Entropie normalisée H (bleu = ordonné, rouge = désordonné)', fontsize=10)

# Annoter les grandes villes
top_cities = df_state.nlargest(12, 'total_cap')
for _, row in top_cities.iterrows():
    ax.annotate(row['city'],
                (np.log1p(row['density']), row['cv']),
                xytext=(6, 4), textcoords='offset points',
                fontsize=8, color='#222222',
                arrowprops=dict(arrowstyle='->', color=GREY, lw=0.8))

# Zones de phase (estimation)
dens_med = np.log1p(df_state['density'].median())
cv_med   = df_state['cv'].median()

ax.axhline(cv_med, color=GREY, ls=':', lw=1.2, alpha=0.7)
ax.axvline(dens_med, color=GREY, ls=':', lw=1.2, alpha=0.7)

# Labels de phases
bbox_style = dict(boxstyle='round,pad=0.3', fc='white', ec=GREY, alpha=0.75)
ax.text(0.05, 0.92, '"Phase gazeuse"\n(épars, hétérogène)',
        transform=ax.transAxes, fontsize=9, color=RED, va='top', bbox=bbox_style)
ax.text(0.55, 0.92, '"Phase liquide critique"\n(dense, hétérogène)',
        transform=ax.transAxes, fontsize=9, color=ORANGE, va='top', bbox=bbox_style)
ax.text(0.05, 0.12, '"Phase solide"\n(épars, uniforme)',
        transform=ax.transAxes, fontsize=9, color=BLUE, va='top', bbox=bbox_style)
ax.text(0.55, 0.12, '"Phase optimale"\n(dense, uniforme)',
        transform=ax.transAxes, fontsize=9, color=GREEN, va='top', bbox=bbox_style)

ax.set_xlabel('log(1 + densité de stations) — analogie Pression log(P)', fontsize=11)
ax.set_ylabel('Coefficient de variation des capacités — analogie Température T', fontsize=11)
ax.set_title('Diagramme d\'état thermodynamique des réseaux VLS français\n'
             '(taille ∝ capacité totale | couleur = entropie normalisée H)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# Légende taille
for cap, label in [(200, '200 empl'), (1000, '1 000 empl'), (3000, '3 000 empl')]:
    sz = cap / df_state['total_cap'].max() * 500 + 30
    ax.scatter([], [], s=sz, c=GREY, alpha=0.6, label=label)
ax.legend(title='Capacité totale', loc='lower right', framealpha=0.85)

plt.tight_layout()
plt.savefig('../data/processed/fig_phase_diagram_vls.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistiques par "phase"
df_state['phase'] = 'transition'
df_state.loc[(df_state['density'] < df_state['density'].median()) & 
              (df_state['cv'] > cv_med), 'phase'] = 'gazeuse'
df_state.loc[(df_state['density'] >= df_state['density'].median()) & 
              (df_state['cv'] > cv_med), 'phase'] = 'liquide'
df_state.loc[(df_state['density'] < df_state['density'].median()) & 
              (df_state['cv'] <= cv_med), 'phase'] = 'solide'
df_state.loc[(df_state['density'] >= df_state['density'].median()) & 
              (df_state['cv'] <= cv_med), 'phase'] = 'optimale'

print('Répartition par phase thermodynamique :')
print(df_state.groupby('phase')[['city']].count().rename(columns={'city':'N villes'}))

---
## 10. Récapitulatif — Tableau de synthèse physique

| Concept physique | Variable bikeshare | Résultat clé |
|---|---|---|
| Entropie de Boltzmann | Distribution des capacités par ville | H médian = 0.85–0.90 → réseaux relativement uniformes |
| Température thermodynamique | T* = ⟨c⟩ / c_max | Réseaux diversifiés : T* ∈ [0.3, 0.8] |
| Distribution de Boltzmann | Distribution nationale des capacités | Ajustement lognormal > exponentiel → contraintes urbanistiques |
| Loi de puissance (scale-free) | Distribution des capacités | α ≈ 2–3 → topologie scale-free partielle |
| Modèle de gravité newtonien | Flux inter-urbains potentiels | Exposant ≈ -2 compatible avec Newton |
| Diffusion de Fick | Rééquilibrage des vélos | Gradient de Fick → recommandations redistribution |
| Percolation | Connectivité réseau | Seuil r_c ≈ 400–600 m pour Montpellier |
| MaxEnt (Jaynes) | Distribution optimale théorique | Divergence KL mesure les contraintes urbanistiques |
| Diagramme de phases | État thermodynamique global | 4 phases identifiées : gazeuse / liquide / solide / optimale |

In [ ]:
# Tableau récapitulatif final
recap = {
    'Concept': [
        'Entropie de Shannon/Boltzmann',
        'Distribution de Boltzmann',
        'Loi de puissance (scale-free)',
        'Modèle de gravité newtonien',
        'Diffusion de Fick',
        'Percolation (Montpellier)',
        'Principe MaxEnt (Jaynes)',
    ],
    'Variable bikeshare': [
        'Distribution des capacités / ville',
        'Distribution nationale des capacités',
        'Distribution des tailles de réseau',
        'Attractivité inter-urbaine (capacité²/distance²)',
        'Gradient spatial de capacité',
        'Connectivité du réseau Vélomagg',
        'Écart à la distribution optimale',
    ],
    'Résultat quantitatif': [
        f'H médian = {df_ent["H_norm"].median():.3f}',
        f'λ = {lambda_fit:.3f} empl⁻¹ ; Lognormal > Exponentiel',
        f'α_MLE = {alpha_mle:.3f}',
        f'Exposant distance = {slope:.2f} (attendu -2.0)',
        f'|gradient| médian = {np.median(np.abs(gradients)):.1f} empl',
        f'r_c = {r_c*1000:.0f} m',
        f'KL médian = {df_maxent["KL_divergence"].median():.4f} nats',
    ]
}

df_recap = pd.DataFrame(recap)
print('=' * 90)
print('SYNTHESE PHYSIQUE — Réseaux VLS français (Gold Standard GBFS)')
print('=' * 90)
print(df_recap.to_string(index=False))
print('=' * 90)
print('\nFossé R. & Pallares G. — 2025-2026')